# 🚀 FAKTA — LSTM Training di Google Colab

**Perubahan:**
- Dataset di-balance 1:1 (undersampling hoax)
- Binary classification (hoax vs valid) — bukan 3-class
- 1 BiLSTM layer, units=32 — anti overfitting
- Embedding dim=64, vocab=10000, dropout=0.5
- Metrics: Macro-F1, per-class recall/precision

## Cara Pakai:
1. Buka https://colab.research.google.com/
2. Klik **Upload** → pilih file ini
3. Runtime → Change runtime type → **T4 GPU**
4. Run semua cell dari atas ke bawah
5. Download model yang sudah di-train dari sidebar Files

---
## Step 1: Install Dependencies & Cek GPU

In [ ]:
# Install dependencies
!pip install -q scikit-learn pandas numpy matplotlib seaborn pyyaml

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"GPU name: {tf.test.gpu_device_name()}")

---
## Step 2: Upload Dataset

In [ ]:
!mkdir -p data/training models/lstm

# Upload manual dari laptop
from google.colab import files
print("Upload file CSV (combined.csv atau dataset lain)...")
uploaded = files.upload()
for filename in uploaded.keys():
    !mv {filename} data/training/
    print(f"  ✅ Moved {filename} to data/training/")

!ls -la data/training/

---
## Step 3: Load & Preprocess Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load semua CSV
csv_files = list(Path('data/training').glob('*.csv'))
print(f"Found {len(csv_files)} CSV file(s)")

dfs = []
for f in csv_files:
    # Coba baca dengan header dulu
    df = pd.read_csv(f)
    
    # Kalau tidak ada kolom text/label, coba tanpa header
    if 'text' not in df.columns and 'label' not in df.columns:
        df = pd.read_csv(f, header=None, names=['text', 'label'])
    
    # Auto-rename kolom
    if 'text' not in df.columns:
        for col in ['tweet', 'content', 'article', 'berita']:
            if col in df.columns:
                df = df.rename(columns={col: 'text'})
                break
    if 'label' not in df.columns:
        for col in ['hoax', 'class', 'is_hoax', 'label']:
            if col in df.columns:
                df = df.rename(columns={col: 'label'})
                break
    
    # Normalize label — BINARY: hoax vs valid (no uncertain)
    df['label'] = df['label'].astype(str).str.lower().str.strip()
    label_map_fn = lambda x: 'hoax' if x in ['hoax', 'false', '1', 'hoaks'] else ('valid' if x in ['valid', 'true', '0', 'tidak hoax', 'non-hoax'] else None)
    df['label'] = df['label'].map(label_map_fn)
    
    df = df[df['label'].isin(['hoax', 'valid'])]
    dfs.append(df[['text', 'label']])
    print(f"  {f.name}: {len(df)} samples ({df['label'].value_counts().to_dict()})")

combined = pd.concat(dfs, ignore_index=True)
combined = combined.dropna(subset=['text', 'label'])
combined = combined.drop_duplicates(subset=['text'])
combined = combined[combined['text'].str.len() > 10]

print(f"\nTotal before balancing: {len(combined)} samples")
print(f"Label distribution:")
print(combined['label'].value_counts())

# === UNDERSAMPLE to 1:1 ===
n_valid = len(combined[combined['label'] == 'valid'])
n_hoax = len(combined[combined['label'] == 'hoax'])

print(f"\nBefore undersampling: hoax={n_hoax}, valid={n_valid}")

if n_hoax > n_valid:
    hoax_df = combined[combined['label'] == 'hoax'].sample(n=n_valid, random_state=42)
    valid_df = combined[combined['label'] == 'valid']
else:
    valid_df = combined[combined['label'] == 'valid'].sample(n=n_hoax, random_state=42)
    hoax_df = combined[combined['label'] == 'hoax']

combined = pd.concat([hoax_df, valid_df], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After undersampling (1:1): hoax={len(hoax_df)}, valid={len(valid_df)}")
print(f"Total: {len(combined)}")

---
## Step 4: Text Cleaning + Normalization

In [ ]:
import re
import random

SLANG_MAP = {
    'yg': 'yang', 'dg': 'dengan', 'dr': 'dari', 'tp': 'tapi',
    'klo': 'kalau', 'kalo': 'kalau', 'gak': 'tidak', 'ga': 'tidak',
    'nggak': 'tidak', 'ngga': 'tidak', 'udh': 'sudah', 'udah': 'sudah',
    'blm': 'belum', 'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'skrg': 'sekarang', 'bs': 'bisa', 'krn': 'karena',
    'utk': 'untuk', 'dpt': 'dapat', 'sm': 'sama', 'jg': 'juga',
    'cuma': 'hanya', 'cm': 'hanya', 'aja': 'saja',
    'sy': 'saya', 'gw': 'saya', 'gue': 'saya',
    'lu': 'kamu', 'loe': 'kamu', 'km': 'kamu',
    'mrk': 'mereka', 'tdk': 'tidak', 'kpn': 'kapan',
    'nih': 'ini', 'tu': 'itu', 'gitu': 'begitu',
}

# Indonesian synonym map for noise injection
SYNONYMS = {
    'menyebabkan': ['memicu', 'mengakibatkan', 'menimbulkan'],
    'menurut': ['berdasarkan', 'mengacu pada'],
    'banyak': ['sejumlah', 'beragam'],
    'besar': ['signifikan', 'luas'],
    'penting': ['krusial', 'vital'],
    'cepat': ['segera'],
    'orang': ['masyarakat', 'warga'],
    'pemerintah': ['pejabat', 'negara'],
    'kesehatan': ['medis'],
    'sudah': ['telah'],
    'tidak': ['belum'],
    'ada': ['terdapat', 'ditemukan'],
    'berita': ['informasi', 'kabar'],
    'meninggal': ['wafat'],
    'bahaya': ['ancaman', 'risiko'],
}

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"()-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_slang(text):
    words = text.split()
    return ' '.join(SLANG_MAP.get(w, w) for w in words)

# === Noise / Augmentation Functions (for training) ===
def noise_synonym_swap(text, rate=0.1):
    words = text.split()
    result = []
    for w in words:
        if w in SYNONYMS and random.random() < rate:
            result.append(random.choice(SYNONYMS[w]))
        else:
            result.append(w)
    return ' '.join(result)

def noise_random_deletion(text, rate=0.08):
    words = text.split()
    if len(words) < 5:
        return text
    kept = [w for w in words if random.random() > rate]
    return ' '.join(kept) if kept else text

def noise_word_shuffle(text, rate=0.05):
    words = text.split()
    if len(words) < 4:
        return text
    n_swaps = max(1, int(len(words) * rate))
    for _ in range(n_swaps):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

def noise_char_level(text, rate=0.02):
    chars = list(text)
    for i in range(len(chars)):
        if chars[i] == ' ' and random.random() < rate * 2:
            chars[i] = ''  # merge two words
    result = ''.join(chars)
    if len(result) < 3:
        return text
    return result

def apply_noise(text):
    """Randomly apply one noise type to text. Used during training."""
    strategy = random.choice(['clean', 'clean', 'swap', 'delete', 'shuffle', 'char'])
    if strategy == 'clean':
        return text
    elif strategy == 'swap':
        return noise_synonym_swap(text, rate=0.1)
    elif strategy == 'delete':
        return noise_random_deletion(text, rate=0.08)
    elif strategy == 'shuffle':
        return noise_word_shuffle(text, rate=0.05)
    else:
        return noise_char_level(text, rate=0.02)

print('Preprocessing...')
combined['cleaned'] = combined['text'].apply(clean_text)
combined['normalized'] = combined['cleaned'].apply(normalize_slang)

# Test noise functions
sample = combined['normalized'].iloc[0]
print(f"Original:  {sample[:100]}")
for i in range(3):
    random.seed(i)
    print(f"Noised {i+1}:   {apply_noise(sample)[:100]}")

---
## Step 5: Split → Tokenize → Prepare Sequences

**PENTING:** Split dulu baru tokenize (bukan sebaliknya). Ini mencegah data leakage.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Config — smaller to prevent overfitting
MAX_WORDS = 10000
MAX_LEN = 200
EMBEDDING_DIM = 64

# Split DULU (70/15/15) — hindari data leakage
train_df, test_df = train_test_split(
    combined, test_size=0.15, random_state=42, stratify=combined['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.18, random_state=42, stratify=train_df['label']
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Train distribution: {train_df['label'].value_counts().to_dict()}")

# Tokenize HANYA dari training data
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['normalized'].tolist())

def make_sequences(texts, labels):
    seq = tokenizer.texts_to_sequences(texts)
    X = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    label_map = {'valid': 0, 'hoax': 1}
    y = np.array([label_map[l] for l in labels])
    return X, y

X_train, y_train = make_sequences(train_df['normalized'], train_df['label'])
X_val, y_val = make_sequences(val_df['normalized'], val_df['label'])
X_test, y_test = make_sequences(test_df['normalized'], test_df['label'])

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Vocabulary: {len(tokenizer.word_index)} words")

# Class weights (should be ~1:1 since we undersampled, but still apply)
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print(f"Class weights: {class_weight}")

---
## Step 6: Build BiLSTM Model (Anti-Overfitting)

Improvement:
- Dropout lebih tinggi (0.5)
- LSTM units lebih kecil (32)
- 1 BiLSTM layer saja
- Noise injection on-the-fly selama training

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.regularizers import l2

# Model config — SMALL to prevent overfitting
EMBEDDING_DIM = 64
LSTM_UNITS = 32
DROPOUT_RATE = 0.5
NUM_CLASSES = 2  # Binary: hoax vs valid

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN,
        mask_zero=True,
    ),
    Dropout(DROPOUT_RATE),
    # SINGLE BiLSTM layer — reduced from 2
    Bidirectional(LSTM(
        LSTM_UNITS,
        return_sequences=False,
        dropout=DROPOUT_RATE,
        # NO recurrent_dropout — biar cuDNN aktif (10x lebih cepat)
    )),
    Dense(16, activation='relu'),
    Dropout(DROPOUT_RATE),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

---
## Step 7: Train Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

# === Data Generator with On-the-Fly Noise Injection ===
# Instead of training on static X_train, we augment text on-the-fly each epoch.
# This prevents the model from memorizing exact training sequences.

def text_to_sequences(texts):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

class NoiseDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, texts, labels, tokenizer, max_len, batch_size, noise_rate=0.6):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.batch_size = batch_size
        self.noise_rate = noise_rate  # probability of applying noise

    def __len__(self):
        return int(np.ceil(len(self.texts) / self.batch_size))

    def __getitem__(self, idx):
        batch_texts = self.texts[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_labels = self.labels[idx * self.batch_size : (idx + 1) * self.batch_size]

        # Apply noise on-the-fly
        noisy_texts = []
        for t in batch_texts:
            if random.random() < self.noise_rate:
                noisy_texts.append(apply_noise(t))
            else:
                noisy_texts.append(t)

        X = text_to_sequences(noisy_texts)
        y = np.array(batch_labels)
        return X, y

    def on_epoch_end(self):
        pass  # shuffle handled by keras fit()

# Create generators
train_gen = NoiseDataGenerator(
    train_df['normalized'].tolist(), y_train,
    tokenizer, MAX_LEN, BATCH_SIZE, noise_rate=0.6
)

callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        'models/lstm/lstm_model.keras', save_best_only=True,
        monitor='val_loss', verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
]

EPOCHS = 20
BATCH_SIZE = 64

print(f"Training with noise injection (rate=0.6)")
print(f"  - 60% of training batches get random noise each epoch")
print(f"  - Model never sees the same exact text twice")
print()

history = model.fit(
    train_gen,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks,
)

---
## Step 8: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

# Info overfitting
train_acc = max(history.history['accuracy'])
val_acc = max(history.history['val_accuracy'])
gap = train_acc - val_acc
print(f"Best train acc: {train_acc:.4f}")
print(f"Best val acc: {val_acc:.4f}")
print(f"Gap (overfitting indicator): {gap:.4f}")
if gap > 0.05:
    print("⚠️ Overfitting terdeteksi (gap > 5%)")
elif gap > 0.02:
    print("ℹ️ Sedikit overfitting (gap 2-5%)")
else:
    print("✅ Tidak ada overfitting signifikan (gap < 2%)")

---
## Step 9: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
import seaborn as sns

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"{'='*50}")
print(f"TEST SET EVALUATION")
print(f"{'='*50}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss:    {loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
labels = ['valid', 'hoax']

# Detailed metrics
macro_f1 = f1_score(y_test, y_pred, average='macro')
per_recall = recall_score(y_test, y_pred, average=None, labels=[0, 1])
per_prec = precision_score(y_test, y_pred, average=None, labels=[0, 1])

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=labels))
print(f"\nMacro-F1:          {macro_f1:.4f}")
print(f"Recall  valid:     {per_recall[0]:.4f}")
print(f"Recall  hoax:      {per_recall[1]:.4f}")
print(f"Precision valid:   {per_prec[0]:.4f}")
print(f"Precision hoax:    {per_prec[1]:.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

---
## Step 10: Quick Inference Test

In [ ]:
---
## Step 10.5: Feature Importance per Kata (TF-IDF + Coefficient Analysis)

Cell ini menganalisis kata mana yang paling berpengaruh dalam membedakan hoax vs valid.
Metode: TF-IDF + Logistic Regression (sebagai proxy feature importance untuk LSTM).

---
## Step 11: Save Model & Tokenizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd
import matplotlib.pyplot as plt

# Build TF-IDF + Logistic Regression pipeline untuk feature importance
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
X_tfidf = tfidf.fit_transform(train_df['normalized'].tolist())

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_tfidf, y_train)

# Get feature names
feature_names = tfidf.get_feature_names_out()

# Coefficients: positif = cenderung hoax, negatif = cenderung valid
coefs = clf.coef_[0]

# Top 30 kata yang paling mengarah ke HOAX
top_hoax_idx = coefs.argsort()[-30:][::-1]
top_hoax = [(feature_names[i], round(float(coefs[i]), 4)) for i in top_hoax_idx]

# Top 30 kata yang paling mengarah ke VALID
top_valid_idx = coefs.argsort()[:30]
top_valid = [(feature_names[i], round(float(coefs[i]), 4)) for i in top_valid_idx]

print("=" * 60)
print("FEATURE IMPORTANCE — Top 30 Kata Paling Hoax")
print("=" * 60)
for word, score in top_hoax:
    print(f"  {word:30s}  {score:+.4f}")

print()
print("=" * 60)
print("FEATURE IMPORTANCE — Top 30 Kata Paling Valid")
print("=" * 60)
for word, score in top_valid:
    print(f"  {word:30s}  {score:+.4f}")

# Plot feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Hoax words
hoax_words = [w for w, _ in top_hoax[::-1]]
hoax_scores = [s for _, s in top_hoax[::-1]]
axes[0].barh(hoax_words, hoax_scores, color='#ef4444', alpha=0.8)
axes[0].set_title('Top Kata Indikator HOAX', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Coefficient (semakin positif = semakin hoax)')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Valid words
valid_words = [w for w, _ in top_valid]
valid_scores = [s for _, s in top_valid]
axes[1].barh(valid_words, valid_scores, color='#22c55e', alpha=0.8)
axes[1].set_title('Top Kata Indikator VALID', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Coefficient (semakin negatif = semakin valid)')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Save to CSV
fi_df = pd.DataFrame({
    'word': feature_names,
    'coefficient': [round(float(c), 4) for c in coefs],
    'importance': ['hoax' if c > 0 else 'valid' for c in coefs],
})
fi_df = fi_df.sort_values('coefficient', ascending=False)
fi_df.to_csv('feature_importance.csv', index=False)
print("\nFeature importance saved to feature_importance.csv")
print(f"Total words analyzed: {len(feature_names)}")

---
## Step 12: Download Model untuk Lokal

In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile('lstm_model.zip', 'w') as zf:
    zf.write('models/lstm/lstm_model.keras')
    zf.write('models/lstm/tokenizer.pkl')
    zf.write('models/lstm/label_map.json')
    zf.write('training_history.png')
    zf.write('confusion_matrix.png')

print("Downloading model package...")
files.download('lstm_model.zip')
print("\nDone! Extract lstm_model.zip dan taruh di:")
print("  models/lstm/")